<a href="https://colab.research.google.com/github/neel0086/MachineLearning/blob/main/bank_marketing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plot
import pandas as pd

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("henriqueyamahata/bank-marketing")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'bank-marketing' dataset.
Path to dataset files: /kaggle/input/bank-marketing


In [ ]:
df = pd.read_csv(os.path.join(path, "bank-additional-full.csv"), sep=';')

print(df.head())
print(df.columns.tolist())

   age        job  marital    education  default housing loan    contact  \
0   56  housemaid  married     basic.4y       no      no   no  telephone   
1   57   services  married  high.school  unknown      no   no  telephone   
2   37   services  married  high.school       no     yes   no  telephone   
3   40     admin.  married     basic.6y       no      no   no  telephone   
4   56   services  married  high.school       no      no  yes  telephone   

  month day_of_week  ...  campaign  pdays  previous     poutcome emp.var.rate  \
0   may         mon  ...         1    999         0  nonexistent          1.1   
1   may         mon  ...         1    999         0  nonexistent          1.1   
2   may         mon  ...         1    999         0  nonexistent          1.1   
3   may         mon  ...         1    999         0  nonexistent          1.1   
4   may         mon  ...         1    999         0  nonexistent          1.1   

   cons.price.idx  cons.conf.idx  euribor3m  nr.employed

In [ ]:
TARGET_COLUMN = "y"

X = df.drop(columns=[TARGET_COLUMN])
Y = df[TARGET_COLUMN].map({
    "no":0,
    "yes":1
})

In [ ]:
indices = np.random.permutation(len(X))

X = X.iloc[indices].reset_index(drop=True)
Y = Y.iloc[indices].reset_index(drop=True)

In [ ]:
#TEST TRAIN Split
split = int(len(X)*0.8)

X_train = X.iloc[:split].copy()
X_test = X.iloc[split:].copy()

Y_train = Y.iloc[:split].copy()
Y_test = Y.iloc[split:].copy()

In [ ]:
categorical_columns = X_train.select_dtypes(include="object").columns

X_train = pd.get_dummies(X_train, columns = categorical_columns)
X_test = pd.get_dummies(X_test, columns = categorical_columns)

In [ ]:
mean = X_train.mean()

std = X_train.std()

X_train = (X_train - mean)/std
X_test = (X_test - mean)/std

X_train = X_train.to_numpy(dtype=float)
X_test = X_test.to_numpy(dtype=float)
Y_train = Y_train.to_numpy(dtype=float)
Y_test = Y_test.to_numpy(dtype=float)

In [ ]:
weights = np.zeros(X_train.shape[1])

bias = 0
learning_rate = 0.01

epochs = 10000

In [ ]:
def sigmoid(z):
  return 1/(1+np.exp(-z))

In [ ]:
def binary_cross_entropy(y_true, y_pred):
  epsilon = 1e-5

  y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
  return np.mean(-(y_true*np.log(y_pred) + (1-y_true)*np.log(1-y_pred)))

In [ ]:
for epoch in range(epochs):
  z = X_train@weights + bias

  prediction = sigmoid(z)

  current_loss = binary_cross_entropy(Y_train, prediction)
  error = prediction - Y_train

  dw = (1/len(X_train))*(X_train.T@error)
  db = (1/len(X_train))*np.mean(error)

  weights -= learning_rate*dw
  bias -= learning_rate*db

  if epoch%500==0:print("Current Loss:",current_loss)


Current Loss: 0.6931471805599453
Current Loss: 0.618480272680119
Current Loss: 0.611133701722922
Current Loss: 0.6091890184428579
Current Loss: 0.6084815107601015
Current Loss: 0.6081516187864463
Current Loss: 0.6079637134896785
Current Loss: 0.6078395910306656
Current Loss: 0.6077471668244033
Current Loss: 0.6076754525995764
Current Loss: 0.6076168041602052
Current Loss: 0.6075667957386607
Current Loss: 0.6075226969249213
Current Loss: 0.6074827514013293
Current Loss: 0.6074457962133669
Current Loss: 0.6074110441598856
Current Loss: 0.6073779524981918
Current Loss: 0.6073461407790991
Current Loss: 0.6073153380859154
Current Loss: 0.607285348488686


In [ ]:
z = X_test @ weights + bias

probability = sigmoid(z)

prediction = (probability >= 0.5).astype(int)

accuracy = np.mean(prediction == Y_test)
print("\n==============================")
print("Accuracy :", accuracy * 100, "%")
print("==============================")


Accuracy : 88.48021364408837 %
